# 2차 — 난자 출처×나이 상호작용 (특정 부분집단 불균형 완화)

## 배경 (EDA로 발견한 패턴)
나이대별 성공률이 단조감소가 아니라 45-50세에서 다시 튀어오르는 이상치가 있었음 → 원인은
**심슨의 역설**: 본인 난자는 나이 들수록 32.6%→2.8%로 무너지는데, **기증 난자는 나이와 거의
무관하게 29~34%로 평평함**. 45-50세 그룹은 기증 비율이 53%까지 치솟아서 평균이 다시 올라간 것.

정자 출처는 이런 패턴이 없음(나이 따라 계속 감소) — 여성 난자 나이가 핵심 변수라는 걸 뒷받침.

## 가설
기증 난자 그룹은 전체의 6.15%뿐이라 나이대별로 쪼개면 표본이 더 적어짐(예: 18-34세+기증=2,871행).
트리 모델이 이 "나이와 무관하게 평평한" 패턴을 충분히 안정적으로 학습 못했을 수 있음 → 1등팀이
말한 "특정 튀는 집단의 불균형 완화"가 이런 종류일 가능성.

## 테스트할 것
1. **baseline** (현재 그대로)
2. **인터랙션 피처 추가**: 나이대×난자출처 결합 카테고리 (트리가 상호작용을 더 쉽게 찾도록)
3. **Sample weight**: 기증 난자 행에 가중치 2x/3x/5x/8x (소수 부분집단에 더 집중하도록 — IVF/DI를
   "분리 모델"로 쪼개는 건 예전에 손해였지만, 통합 모델 안에서 가중치만 주는 건 다른 접근)
4. 둘 다 결합

## 누수 방지
이 실험은 test를 안 씀 (CV 검증 전용), 인터랙션 피처도 train 정보만으로 구성 — 안전.

---
**저장 위치:** `experiment_history/2차/2차_subgroup_imbalance_egg_source.ipynb`


In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import warnings

warnings.filterwarnings("ignore")


In [2]:
DATA_DIR = Path("../../data")
SHARED_DIR = Path("..")
TRAIN_PATH = DATA_DIR / "train.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]

ROBUST_PARAMS_PATH = SHARED_DIR / "lgbm_robust_best_params.json"
FALLBACK_PARAMS_PATH = SHARED_DIR / "best_params.json"

N_SPLITS = 5
SEEDS_STAGE1 = [1004, 42, 7]
SEEDS_STAGE2 = [1004, 42, 7, 123, 2024, 88, 555, 999, 31, 77]
STAGE2_PROMOTION_THRESHOLD = 0.0002


## 1. 데이터 로드 + 전처리 (+ 인터랙션 피처 추가)

In [3]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    # ★ 새 인터랙션 피처: 나이대 x 난자출처 결합 카테고리
    df["age_x_egg_source"] = df["시술 당시 나이"].astype(str) + "_" + df["난자 출처"].astype(str)
    return df


def fill_na(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


df_train = fill_na(base_features(train_raw.copy()).drop(columns=DEAD_COLS))
cat_cols = df_train.select_dtypes(include=["object"]).columns.tolist()

for col in cat_cols:
    le = LabelEncoder()
    df_train[col] = le.fit_transform(df_train[col].astype(str))

y = df_train[TARGET_COL].values.astype(int)
X_no_interaction = df_train.drop(columns=[TARGET_COL, "age_x_egg_source"])
X_with_interaction = df_train.drop(columns=[TARGET_COL])

donor_egg_mask = (train_raw["난자 출처"] == "기증 제공").values
print(f"기증 난자 비율: {donor_egg_mask.mean():.2%} ({donor_egg_mask.sum()}행)")
print(f"train shape (인터랙션 없음): {X_no_interaction.shape}, (인터랙션 포함): {X_with_interaction.shape}")


기증 난자 비율: 6.15% (15769행)
train shape (인터랙션 없음): (256351, 67), (인터랙션 포함): (256351, 68)


## 2. 시작 파라미터 로드

In [4]:
def load_base_params(path: Path) -> dict:
    with open(path, encoding="utf-8") as f:
        loaded = json.load(f)
    return loaded.get("best_params", loaded) if isinstance(loaded, dict) else loaded

params_path = ROBUST_PARAMS_PATH if ROBUST_PARAMS_PATH.exists() else FALLBACK_PARAMS_PATH
base_params = load_base_params(params_path)
print(f"파라미터 출처: {params_path}")


파라미터 출처: ../lgbm_robust_best_params.json


## 3. 평가 함수

In [5]:
def evaluate_config(seeds, X, sample_weight_mask=None, weight_multiplier=1.0):
    """sample_weight_mask: True인 행에 weight_multiplier를 곱함 (나머지는 weight=1.0)"""
    seed_aucs = []
    oof_per_seed = []
    for seed in seeds:
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        oof = np.zeros(len(y))
        for tr_idx, val_idx in skf.split(X, y):
            X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
            X_val = X.iloc[val_idx]

            sw = None
            if sample_weight_mask is not None:
                sw = np.ones(len(tr_idx))
                sw[sample_weight_mask[tr_idx]] = weight_multiplier

            m = LGBMClassifier(**base_params, random_state=seed, verbose=-1)
            m.fit(X_tr, y_tr, sample_weight=sw)
            oof[val_idx] = m.predict_proba(X_val)[:, 1]

        seed_aucs.append(roc_auc_score(y, oof))
        oof_per_seed.append(oof)

    oof_per_seed = np.array(oof_per_seed)
    bagged_auc = roc_auc_score(y, oof_per_seed.mean(axis=0))
    return np.array(seed_aucs), bagged_auc


## 4. Stage 1 — 3시드 빠른 탐색

In [6]:
configs = [
    ("baseline", X_no_interaction, None, 1.0),
    ("인터랙션 피처만", X_with_interaction, None, 1.0),
    ("기증난자 weight=2x", X_no_interaction, donor_egg_mask, 2.0),
    ("기증난자 weight=3x", X_no_interaction, donor_egg_mask, 3.0),
    ("기증난자 weight=5x", X_no_interaction, donor_egg_mask, 5.0),
    ("기증난자 weight=8x", X_no_interaction, donor_egg_mask, 8.0),
    ("인터랙션+weight=3x", X_with_interaction, donor_egg_mask, 3.0),
]

print(f"Stage 1 탐색 (시드 {len(SEEDS_STAGE1)}개: {SEEDS_STAGE1})\n")
print(f"{'설정':<24} | {'평균AUC':>8} | {'표준편차':>8} | {'베이스라인差':>12} | 시간")
print("-" * 72)

stage1_results = {}
baseline_mean = None
for label, X_cfg, mask, mult in configs:
    t0 = time.time()
    aucs, _ = evaluate_config(SEEDS_STAGE1, X_cfg, mask, mult)
    mean_auc, std_auc = aucs.mean(), aucs.std()
    if label == "baseline":
        baseline_mean = mean_auc
    diff = mean_auc - baseline_mean if baseline_mean is not None else 0.0
    stage1_results[label] = {"X": X_cfg, "mask": mask, "mult": mult, "mean": mean_auc, "std": std_auc}
    print(f"{label:<24} | {mean_auc:>8.5f} | {std_auc:>8.5f} | {diff:>+12.5f} | {time.time()-t0:>4.0f}s")


Stage 1 탐색 (시드 3개: [1004, 42, 7])

설정                       |    평균AUC |     표준편차 |       베이스라인差 | 시간
------------------------------------------------------------------------
baseline                 |  0.73991 |  0.00005 |     +0.00000 |   51s
인터랙션 피처만                 |  0.73993 |  0.00002 |     +0.00002 |   48s
기증난자 weight=2x           |  0.73985 |  0.00012 |     -0.00006 |   50s
기증난자 weight=3x           |  0.73964 |  0.00013 |     -0.00027 |   51s
기증난자 weight=5x           |  0.73916 |  0.00011 |     -0.00075 |   55s
기증난자 weight=8x           |  0.73835 |  0.00011 |     -0.00156 |   61s
인터랙션+weight=3x           |  0.73965 |  0.00008 |     -0.00026 |   53s


## 5. Stage 2 — baseline보다 분명히 나은 것만 10시드 정밀검증

In [7]:
candidates = [
    label for label, r in stage1_results.items()
    if label != "baseline" and (r["mean"] - baseline_mean) > STAGE2_PROMOTION_THRESHOLD
]
print(f"Stage 2 정밀검증 후보: {candidates if candidates else '없음'}\n")

print(f"{'설정':<24} | {'10시드 평균':>10} | {'표준편차':>8} | {'10시드 배깅AUC':>13}")
print("-" * 68)

for label in ["baseline"] + candidates:
    r = stage1_results[label]
    aucs2, bagged2 = evaluate_config(SEEDS_STAGE2, r["X"], r["mask"], r["mult"])
    print(f"{label:<24} | {aucs2.mean():>10.5f} | {aucs2.std():>8.5f} | {bagged2:>13.5f}")

print()
print("비교 기준:")
print("  10시드 배깅 baseline: 0.74036 (기존) / 0.74037 (견고탐색)")
print("  노이즈 표준편차(참고): 0.00011")


Stage 2 정밀검증 후보: 없음

설정                       |    10시드 평균 |     표준편차 |    10시드 배깅AUC
--------------------------------------------------------------------
baseline                 |    0.73997 |  0.00013 |       0.74037

비교 기준:
  10시드 배깅 baseline: 0.74036 (기존) / 0.74037 (견고탐색)
  노이즈 표준편차(참고): 0.00011


## 6. 해석 가이드

- baseline 행이 0.74036~0.74037 근처로 나오면 재현성 확인됨
- 후보 중 **10시드 배깅 AUC가 baseline보다 노이즈(0.00011)의 2~3배 이상 높으면** 진짜 개선
- 인터랙션 피처만 추가한 게 효과 없다면 "트리가 이미 이 상호작용을 충분히 잘 찾고 있다"는 뜻
  (established pattern: 대부분의 manual FE가 무효했던 것과 같은 이유)
- sample_weight 쪽이 더 잘 작동한다면, 이게 1등팀이 말한 "부분집단 불균형 완화"와 같은 종류의
  레버일 가능성 — 다른 소수 부분집단(예: 6회 이상 시술군)에도 같은 방식 적용해볼 가치 있음
